In [13]:
import os
import glob
import torch
import pandas as pd
import numpy as np
from tqdm import tqdm

# --- 좌표 변환 헬퍼 함수 ---
def global_to_local_numpy(positions, ref_pos):
    dx = positions[..., 0] - ref_pos[..., 0]
    dy = positions[..., 1] - ref_pos[..., 1]
    dpsi = positions[..., 2] - ref_pos[..., 2] 
    
    psi_std = ref_pos[..., 2]
    cos_p = np.cos(psi_std)
    sin_p = np.sin(psi_std)
    
    loc_x = dx * cos_p + dy * sin_p
    loc_y = -dx * sin_p + dy * cos_p
    return np.stack((loc_x, loc_y, dpsi), axis=-1)

# --- 1. 파일 목록 확보 ---
folders = ["C:/workspace/carsim_processed_data_25hz"]
all_files = []
for f in folders: 
    all_files.extend(glob.glob(os.path.join(f, "*.csv")))

seq_len = 4
stride = 1

# 🌟 원하는 Horizon 스텝 설정 (4든 8이든 여기만 바꾸면 연동됩니다)
num_steps = 10
lookahead_margin = num_steps + 1  

# --- 2. 통계량(mean, std) 계산 ---
all_samples, all_accels = [], []

print("슬라이딩 윈도우 기반 글로벌 표준화 파라미터 계산 중...")
for f in tqdm(all_files, desc="통계량 산출 CSV 분석"):
    df = pd.read_csv(f)
    states = df[['vx', 'vy', 'dpsi']].values
    ctrls = df[['Throttle', 'Brake', 'delta']].values
    positions = df[['X', 'Y', 'psi']].values
    accels = df[['ax', 'ay']].values
    
    for start_idx in range(0, len(states) - seq_len - lookahead_margin, stride):
        seq_pos = positions[start_idx : start_idx + seq_len]
        mid_idx = seq_len // 2
        ref_pos = seq_pos[mid_idx]
        
        loc_pos = global_to_local_numpy(seq_pos, ref_pos)
        sample = np.hstack((loc_pos, states[start_idx : start_idx + seq_len], ctrls[start_idx : start_idx + seq_len]))
        all_samples.append(sample)
        all_accels.append(accels[start_idx : start_idx + seq_len])
        
concat_inp = np.vstack(all_samples)
concat_acc = np.vstack(all_accels)

global_stats = {
    'in_mean': concat_inp.mean(axis=0), 
    'in_std': concat_inp.std(axis=0),
    'accel_std': concat_acc.std(axis=0)
}
print("글로벌 데이터 통계량 계산 완료.")

# --- 3. 로컬 텐서 리스트 채우기 ---
in_mean_t = torch.tensor(global_stats['in_mean'], dtype=torch.float32)
in_std_t = torch.tensor(global_stats['in_std'], dtype=torch.float32)

all_processed_inputs, all_curr_raw, all_ctrl, all_accel = [], [], [], []

# 동적 저장을 위한 딕셔너리 컨테이너
target_containers = {f't{i}': [] for i in range(1, num_steps + 1)}          
ctrl_containers = {f'ctrl_t{i}': [] for i in range(1, num_steps)}          
all_final_pos = []                                                            

all_ref_pos = []         
all_curr_global_pos = [] 

def local_transform(pos, r_pos):
    dx = pos[..., 0] - r_pos[..., 0]
    dy = pos[..., 1] - r_pos[..., 1]
    dpsi = pos[..., 2] - r_pos[..., 2]
    
    psi_std = r_pos[..., 2]
    cos_p = torch.cos(psi_std)
    sin_p = torch.sin(psi_std)
    
    return torch.stack((dx * cos_p + dy * sin_p, -dx * sin_p + dy * cos_p, dpsi), dim=-1)

for f in tqdm(all_files, desc="최종 데이터셋 텐서 빌드 및 정규화"):
    df = pd.read_csv(f)
    states = torch.tensor(df[['vx', 'vy', 'dpsi']].values, dtype=torch.float32)
    accels = torch.tensor(df[['ax', 'ay']].values, dtype=torch.float32)
    positions = torch.tensor(df[['X', 'Y', 'psi']].values, dtype=torch.float32)
    ctrl_inputs = torch.tensor(df[['Throttle', 'Brake', 'delta']].values, dtype=torch.float32)

    for start_idx in range(0, len(states) - seq_len - lookahead_margin, stride):
        curr_idx = start_idx + seq_len - 1
        mid_idx = start_idx + seq_len // 2
        
        ref_pos = positions[mid_idx]
        curr_global_pos = positions[curr_idx] 
        
        loc_pos_seq = local_transform(positions[start_idx : start_idx + seq_len], ref_pos)
        
        in_unnorm = torch.cat([loc_pos_seq, states[start_idx : start_idx + seq_len], ctrl_inputs[start_idx : start_idx + seq_len]], dim=1)
        all_processed_inputs.append((in_unnorm - in_mean_t) / (in_std_t + 1e-6))
        
        curr_pos_loc = local_transform(positions[curr_idx], ref_pos)
        all_curr_raw.append(torch.cat([states[curr_idx], curr_pos_loc]))
        all_ctrl.append(ctrl_inputs[curr_idx])
        all_accel.append(accels[curr_idx])
        
        for i in range(1, num_steps + 1):
            ti_pos_loc = local_transform(positions[curr_idx + i], ref_pos)
            target_containers[f't{i}'].append(
                torch.cat([states[curr_idx + i], ti_pos_loc, accels[curr_idx + i]])
            )
            
        for i in range(1, num_steps):
            ctrl_containers[f'ctrl_t{i}'].append(ctrl_inputs[curr_idx + i])
            
        # 🎯 최외각 시점의 위치 정보
        final_pos_loc = local_transform(positions[curr_idx + lookahead_margin], ref_pos)
        all_final_pos.append(final_pos_loc)
        
        all_ref_pos.append(ref_pos)
        all_curr_global_pos.append(curr_global_pos)

# --- 4. 딕셔너리로 저장 ---
dataset_dict = {
    'processed_inputs': torch.stack(all_processed_inputs),
    'curr_raw': torch.stack(all_curr_raw),
    'ctrl': torch.stack(all_ctrl),
    'accel': torch.stack(all_accel),
    f't{num_steps+1}_pos': torch.stack(all_final_pos),  # 🌟 키 이름을 동적으로 매핑 (ex: t5_pos)
    'ref_pos': torch.stack(all_ref_pos),                  
    'curr_global_pos': torch.stack(all_curr_global_pos),  
    'global_stats': global_stats          
}

for k, v in target_containers.items():
    dataset_dict[k] = torch.stack(v)
for k, v in ctrl_containers.items():
    dataset_dict[k] = torch.stack(v)

torch.save(dataset_dict, 'processed_dataset_25hz_carsim.pt')
print(f"🎉 모든 전처리 완료 및 'processed_dataset.pt' 저장 성공! (Horizon: {num_steps} Steps 대응)")

슬라이딩 윈도우 기반 글로벌 표준화 파라미터 계산 중...


통계량 산출 CSV 분석: 100%|██████████| 9/9 [00:00<00:00,  9.79it/s]


글로벌 데이터 통계량 계산 완료.


최종 데이터셋 텐서 빌드 및 정규화: 100%|██████████| 9/9 [00:42<00:00,  4.70s/it]


🎉 모든 전처리 완료 및 'processed_dataset.pt' 저장 성공! (Horizon: 10 Steps 대응)


In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import scipy.io as sio
from tqdm import tqdm
import numpy as np

max_num_steps = 10
m = 1270
lf = 1.015
lr = 1.895
dt = 0.04

# --- 1. 가변 Horizon 동적 빌드형 Dataset ---
class PreProcessedVehicleDataset(Dataset):
    def __init__(self, data_dict):
        self.max_num_steps = max_num_steps
        
        self.processed_inputs = data_dict['processed_inputs'].clone().detach().float()
        self.curr_raw = data_dict['curr_raw'].clone().detach().float()
        self.ctrl = data_dict['ctrl'].clone().detach().float()
        self.accel = data_dict['accel'].clone().detach().float()
        self.ref_pos = data_dict['ref_pos'].clone().detach().float()
        self.curr_global_pos = data_dict['curr_global_pos'].clone().detach().float()
        
        # [동적 매핑] max_num_steps(10) 까지 모두 스택하여 보유
        targets_list = []
        for i in range(1, self.max_num_steps + 1):
            targets_list.append(data_dict[f't{i}'].clone().detach().float())
        self.future_targets = torch.stack(targets_list, dim=1) 
        
        # 가장 마지막 시점(t11)의 위치 정보
        final_pos_key = f't{self.max_num_steps+1}_pos'
        if final_pos_key in data_dict:
            self.final_pos = data_dict[final_pos_key].clone().detach().float()
        else:
            self.final_pos = data_dict[f't{self.max_num_steps+1}'].clone().detach().float()[:, 3:6]
            
        # 미래 제어 입력 동적 바인딩 (ctrl_t1 ~ ctrl_t9)
        if self.max_num_steps > 1:
            ctrls_list = []
            for i in range(1, self.max_num_steps):
                ctrls_list.append(data_dict[f'ctrl_t{i}'].clone().detach().float())
            self.future_ctrls = torch.stack(ctrls_list, dim=1) 
        else:
            self.future_ctrls = torch.empty(len(self.processed_inputs), 0, 3)

    def __len__(self): 
        return len(self.processed_inputs)
        
    def __getitem__(self, idx):
        return (self.processed_inputs[idx], self.curr_raw[idx], self.ctrl[idx], self.accel[idx], 
                self.future_targets[idx], self.final_pos[idx], self.future_ctrls[idx], 
                self.ref_pos[idx], self.curr_global_pos[idx])

# --- 2. 좌표 변환 (Tensor) ---
def global_to_local_torch(positions, ref_pos):
    dx = positions[..., 0] - ref_pos[..., 0]
    dy = positions[..., 1] - ref_pos[..., 1]
    dpsi = positions[..., 2] - ref_pos[..., 2]
    
    psi_std = ref_pos[..., 2]
    cos_p = torch.cos(psi_std)
    sin_p = torch.sin(psi_std)
    return torch.stack((dx * cos_p + dy * sin_p, -dx * sin_p + dy * cos_p, dpsi), dim=-1)

# --- 3. 네트워크 모델 ---
class FullyConnectedBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(nn.Linear(dim, dim), nn.Tanh(), nn.Dropout(0.1))
    def forward(self, x): return self.block(x)

class DeepDynamicsModel(nn.Module):
    def __init__(self, input_dim=9, hidden_dim=256):
        super(DeepDynamicsModel, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, batch_first=True)
        layers = [nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Dropout(0.1)]
        for _ in range(1): layers.append(FullyConnectedBlock(hidden_dim))
        self.mlp_backbone = nn.Sequential(*layers)
        self.output_layer = nn.Linear(hidden_dim, 18)
        self.sigmoid = nn.Sigmoid()
        
        self.register_buffer('t_min', torch.tensor([1, 0.5, 500, -3.0, -0.1, -500.0]))
        self.register_buffer('t_max', torch.tensor([30, 3, 40000.0, 0, 0.1, 500.0]))
        self.register_buffer('dr_min', torch.tensor([500, 0.0, 150.0, 0.2, 500]))
        self.register_buffer('dr_max', torch.tensor([40000.0, 1, 1700.0, 1.7, 40000]))
        self.register_buffer('Iz_min', torch.tensor([500.0]))
        self.register_buffer('Iz_max', torch.tensor([2000.0]))
        
    def forward(self, input_seq, h_0=None):
        _, h_n = self.gru(input_seq, h_0)
        x = self.mlp_backbone(h_n[-1])
        out = self.sigmoid(self.output_layer(x))
        tf = self.t_min + (self.t_max - self.t_min) * out[:, 0:6]
        tr = self.t_min + (self.t_max - self.t_min) * out[:, 6:12]
        dr = self.dr_min + (self.dr_max - self.dr_min) * out[:, 12:17]
        Iz = self.Iz_min + (self.Iz_max - self.Iz_min) * out[:, 17:18]
        return tf, tr, dr, Iz, h_n


def position_step(state, pos):
    vx, vy, r = state[:, 0], state[:, 1], state[:, 2]
    X, Y, psi = pos[:, 0], pos[:, 1], pos[:, 2]
    
    return torch.stack([X + (vx * torch.cos(psi) - vy * torch.sin(psi)) * dt,
                        Y + (vx * torch.sin(psi) + vy * torch.cos(psi)) * dt,
                        psi + r * dt], dim=1)

def velocity_step(state, ctrl, tf, tr, dr, iz):
    vx, vy, r = state[:, 0], state[:, 1], state[:, 2]
    T, brake, delta = ctrl[:, 0], ctrl[:, 1], ctrl[:, 2]

    Frx = (dr[:, 0] - dr[:, 1] * vx) * T -  dr[:,4] * brake - dr[:, 2] - dr[:, 3] * (vx**2)
    af = delta - torch.atan2(r * lf + vy, vx + 1e-6) + tf[:, 4]
    ar = torch.atan2(r * lr - vy, vx + 1e-6) + tr[:, 4]
    Ffy = tf[:, 5] + tf[:, 2] * torch.sin(tf[:, 1] * torch.atan(tf[:, 0]*af - tf[:, 3]*(tf[:, 0]*af - torch.atan(tf[:, 0]*af))))
    Fry = tr[:, 5] + tr[:, 2] * torch.sin(tr[:, 1] * torch.atan(tr[:, 0]*ar - tr[:, 3]*(tr[:, 0]*ar - torch.atan(tr[:, 0]*ar))))
    vx_dot = (- Ffy * torch.sin(delta) + Frx) / m + vy * r
    vy_dot = (Ffy * torch.cos(delta) + Fry) / m - vx * r
    r_dot = (Ffy * torch.cos(delta) * lf - Fry * lr) / (iz.squeeze() + 1e-6)
    return torch.stack([vx + vx_dot * dt, vy + vy_dot * dt, r + r_dot * dt], dim=1), torch.stack([vx_dot - vy * r, vy_dot + vx * r], dim=1)

# --- 5. 커리큘럼 동적 Loss 계산 모듈 ---
def calculate_physics_loss(model, input_seq, curr_raw, curr_ctrl, curr_accel, 
                           batch_targets, batch_final_pos, batch_future_ctrls, gpu_stats, ref_pos, curr_global_pos, active_num_steps):
    
    s_std = gpu_stats['s_std']
    p_std = gpu_stats['p_std']
    a_std = gpu_stats['a_std']
    state_weights = gpu_stats['state_weights']
    pos_weights = gpu_stats['pos_weights']

    tf, tr, dr, iz, _ = model(input_seq)

    loss_tensors = {}
    st_curr = curr_raw[:, 0:3]
    pos_global_curr = curr_global_pos
    ctrl_curr = curr_ctrl

    # --- Dynamic Unrolling 루프 ---
    for i in range(active_num_steps):
        st_next, acc_curr_p = velocity_step(st_curr, ctrl_curr, tf, tr, dr, iz)
        pos_global_next = position_step(st_curr, pos_global_curr)

        # 1. 가속도 오차 계산
        target_accel = curr_accel if i == 0 else batch_targets[:, i-1, 6:8]
        loss_tensors[f'accel_t{i}'] = torch.mean(((acc_curr_p - target_accel) / a_std)**2)

        # 2. 속도(상태) 변수 오차 계산
        target_state = batch_targets[:, i, 0:3]
        loss_tensors[f'state_t{i+1}'] = torch.mean((state_weights * (st_next - target_state) / s_std)**2)

        # 3. 위치 오차 계산
        if i > 0:
            pos_local_next = global_to_local_torch(pos_global_next, ref_pos)
            target_pos = batch_targets[:, i, 3:6]
            loss_tensors[f'pos_t{i+1}'] = torch.mean((pos_weights * (pos_local_next - target_pos) / p_std)**2)

        st_curr = st_next
        pos_global_curr = pos_global_next
        if i < active_num_steps - 1:
            ctrl_curr = batch_future_ctrls[:, i, :]

    # --- Extra: 마지막 시점 위치 오차 ---
    pos_global_final = position_step(st_curr, pos_global_curr)
    pos_local_final = global_to_local_torch(pos_global_final, ref_pos)
    
    if active_num_steps < max_num_steps:
        target_final_pos = batch_targets[:, active_num_steps, 3:6] 
    else:
        target_final_pos = batch_final_pos
        
    loss_tensors[f'pos_t{active_num_steps+1}'] = torch.mean((pos_weights * (pos_local_final - target_final_pos) / p_std)**2)

    
    # --- 시점 가중치 통합 손실 계산 ---
    w_s, w_p, w_a = 1.0, 1.0, 1
    total_loss = 0.0
    
    for i in range(active_num_steps):
        l_s = loss_tensors[f'state_t{i+1}']
        l_a = loss_tensors[f'accel_t{i}']
        l_p = loss_tensors[f'pos_t{i+2}']
        
        # 모든 시점 가중치는 항상 균등하게 1.0 반영
        total_loss += w_s * l_s + w_p * l_p + w_a * l_a

    # 가중치 총합으로 정규화
    total_loss = total_loss / active_num_steps
    '''

    # 현재 활성화된 시퀀스의 가장 마지막 인덱스 지정
    last_idx = active_num_steps - 1
    
    l_s = loss_tensors[f'state_t{last_idx+1}']  # 마지막 스텝의 상태 오차
    l_a = loss_tensors[f'accel_t{last_idx}']    # 마지막 스텝의 가속도 오차
    l_p = loss_tensors[f'pos_t{last_idx+2}']    # 마지막 스텝의 위치 오차 (pos_global_final 기반)
    
    # 오직 마지막 스텝의 손실 항목들만 더하여 최종 loss 생성 (시점 수로 나누지 않음)
    total_loss = w_s * l_s + w_p * l_p + w_a * l_a
    '''

    metrics = {k: v.item() for k, v in loss_tensors.items()}
    return total_loss, metrics


# --- 6. 메인 학습 루프 ---
def main_train(num_epochs=120, batch_size=256):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"현재 연산 장치: {device} | 최대 확보 스텝 수: {max_num_steps}")
    
    print("Pre-processed 데이터 로딩 중...")
    data_dict = torch.load('processed_dataset_25hz_carsim.pt', map_location='cpu', weights_only=False)
    global_stats = data_dict['global_stats']
    
    in_std_t = torch.tensor(global_stats['in_std'], device=device).float()
    accel_std_t = torch.tensor(global_stats['accel_std'], device=device).float()
    
    gpu_stats = {
        's_std': in_std_t[3:6] + 1e-6,
        'p_std': in_std_t[0:3] + 1e-6,
        'a_std': torch.clamp(accel_std_t, min=1.0),
        'state_weights': torch.tensor([1.0, 1, 1], device=device).float(),
        'pos_weights': torch.tensor([1.0, 1.0, 1], device=device).float()
    }
    
    full_dataset = PreProcessedVehicleDataset(data_dict)
    v_idx = int(len(full_dataset) * 0.8)
    train_dataset = Subset(full_dataset, range(0, v_idx))
    val_dataset = Subset(full_dataset, range(v_idx, len(full_dataset)))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, pin_memory=True)
    print(f"데이터 로드 완료! Train 샘플: {len(train_dataset)}, Val 샘플: {len(val_dataset)}")

    model = DeepDynamicsModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)
    
    # 🌟 커리큘럼 러닝 제어를 위한 초기 변수
    best_loss = float('inf')
    active_num_steps = 4  # 초기 시작 스텝

    for epoch in range(num_epochs):
        display_epoch = epoch + 1  # 1부터 시작하는 직관적인 에폭 기준
        
        # --- 🌟 [새로운 커리큘럼] 각 스텝별 가변 에폭 누적 구간 설정 ---
        if display_epoch <= 30:
            target_steps = 4  
        elif display_epoch <= 55:
            target_steps = 5 
        elif display_epoch <= 75:
            target_steps = 6
        elif display_epoch <= 90:
            target_steps = 7 
        elif display_epoch <= 100:
            target_steps = 8   
        elif display_epoch <= 110:
            target_steps = 9  
        else:
            target_steps = 10 
        

        # --- 🌟 페이즈 전환 (Curriculum UP) 감지 및 최고기록 무한대(inf) 초기화 ---
        if target_steps > active_num_steps:
            print("-" * 60)
            print(f"🔥 [Curriculum Phase UP!] 난이도 상승 (Epoch {display_epoch})")
            print(f"👉 예측 스텝 수 증가: {active_num_steps} -> {target_steps}")
            print("👉 새로운 환경에 맞춰 best_loss를 inf로 강제 초기화합니다.")
            print("-" * 60)
            active_num_steps = target_steps
            best_loss = float('inf')  # 이전 페이즈의 최고 기록 무효화

        model.train()
        t_loss = 0
        
        pbar_desc = f"Epoch {display_epoch} (Steps: {active_num_steps})"
        for batch in tqdm(train_loader, desc=pbar_desc):
            i_s, c_r, ct_r, a_r, b_targets, b_final_pos, b_ctrls, ref_pos, curr_global_pos = [b.to(device, non_blocking=True) for b in batch]
            
            optimizer.zero_grad()
            loss, _ = calculate_physics_loss(model, i_s, c_r, ct_r, a_r, b_targets, b_final_pos, b_ctrls, gpu_stats, ref_pos, curr_global_pos, active_num_steps)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            t_loss += loss.item()

        # --- Validation ---
        model.eval()
        v_loss = 0
        v_metrics = {}
        
        with torch.no_grad():
            for batch in val_loader:
                i_s, c_r, ct_r, a_r, b_targets, b_final_pos, b_ctrls, ref_pos, curr_global_pos = [b.to(device, non_blocking=True) for b in batch]
                loss, details = calculate_physics_loss(model, i_s, c_r, ct_r, a_r, b_targets, b_final_pos, b_ctrls, gpu_stats, ref_pos, curr_global_pos, active_num_steps)
                v_loss += loss.item()
                for k, v in details.items(): 
                    v_metrics[k] = v_metrics.get(k, 0.0) + v
        
        avg_v, avg_t = v_loss / len(val_loader), t_loss / len(train_loader)
        num_v = len(val_loader)
        print(f"\n[Epoch {display_epoch} | Unroll: {active_num_steps}] Train Loss: {avg_t:.6f}, Val Loss: {avg_v:.6f}")
        
        accel_strs = [f"t{i}:{v_metrics[f'accel_t{i}']/num_v:.4f}" for i in range(active_num_steps)]
        state_strs = [f"t{i+1}:{v_metrics[f'state_t{i+1}']/num_v:.4f}" for i in range(active_num_steps)]
        pos_strs = [f"t{i+2}:{v_metrics[f'pos_t{i+2}']/num_v:.4f}" for i in range(active_num_steps)]
        
        print(f"   ⚙️ Accel Log -> {', '.join(accel_strs)}")
        print(f"   📈 State Log -> {', '.join(state_strs)}")
        print(f"   📍 Pos Log   -> {', '.join(pos_strs)}")
        
        scheduler.step(avg_v)
        
        # --- 🌟 Checkpointer: 현재 스텝별 최고 기록 비교 및 저장 ---
        if avg_v < best_loss:
            best_loss = avg_v
            
            pth_filename = f'deep_dynamics_best_step_{active_num_steps}.pth'
            mat_filename = f'deep_dynamics_weights_real_step_{active_num_steps}_carsim.mat'
            
            torch.save(model.state_dict(), pth_filename)
            weights = {k.replace('.', '_'): v.detach().cpu().numpy() for k, v in model.state_dict().items()}
            weights.update({'in_mean': global_stats['in_mean'], 'in_std': global_stats['in_std']})
            sio.savemat(mat_filename, weights)
            print(f"🌟 [Best Updated] Val Loss: {avg_v:.6f} -> Saved to {pth_filename}")

if __name__ == "__main__":
    main_train()

현재 연산 장치: cuda | 최대 확보 스텝 수: 10
Pre-processed 데이터 로딩 중...
데이터 로드 완료! Train 샘플: 53899, Val 샘플: 13475


Epoch 1 (Steps: 4): 100%|██████████| 211/211 [00:07<00:00, 29.87it/s]



[Epoch 1 | Unroll: 4] Train Loss: 1.192487, Val Loss: 0.701256
   ⚙️ Accel Log -> t0:0.8418, t1:0.4042, t2:0.2872, t3:0.2300
   📈 State Log -> t1:0.0267, t2:0.0582, t3:0.0921, t4:0.1256
   📍 Pos Log   -> t2:0.1561, t3:0.2446, t4:0.3041, t5:0.3457
🌟 [Best Updated] Val Loss: 0.701256 -> Saved to deep_dynamics_best_step_4.pth


Epoch 2 (Steps: 4): 100%|██████████| 211/211 [00:06<00:00, 31.68it/s]



[Epoch 2 | Unroll: 4] Train Loss: 0.386443, Val Loss: 0.617879
   ⚙️ Accel Log -> t0:0.8309, t1:0.3803, t2:0.2418, t3:0.1996
   📈 State Log -> t1:0.0239, t2:0.0570, t3:0.0898, t4:0.1232
   📍 Pos Log   -> t2:0.1528, t3:0.2299, t4:0.2742, t5:0.2951
🌟 [Best Updated] Val Loss: 0.617879 -> Saved to deep_dynamics_best_step_4.pth


...


[Epoch 118 | Unroll: 10] Train Loss: 0.418786, Val Loss: 0.758001
   ⚙️ Accel Log -> t0:1.4901, t1:0.3323, t2:0.1888, t3:0.1417, t4:0.1215, t5:0.1067, t6:0.1036, t7:0.1070, t8:0.1198, t9:0.1384
   📈 State Log -> t1:0.0517, t2:0.0782, t3:0.0946, t4:0.1076, t5:0.1197, t6:0.1325, t7:0.1467, t8:0.1632, t9:0.1818, t10:0.2028
   📍 Pos Log   -> t2:0.1428, t3:0.2302, t4:0.3040, t5:0.3516, t6:0.3651, t7:0.3506, t8:0.3206, t9:0.3001, t10:0.3198, t11:0.4168
🌟 [Best Updated] Val Loss: 0.758001 -> Saved to deep_dynamics_best_step_10.pth


Epoch 119 (Steps: 10): 100%|██████████| 211/211 [00:12<00:00, 17.14it/s]



[Epoch 119 | Unroll: 10] Train Loss: 0.417560, Val Loss: 0.757307
   ⚙️ Accel Log -> t0:1.4859, t1:0.3320, t2:0.1886, t3:0.1415, t4:0.1214, t5:0.1066, t6:0.1035, t7:0.1069, t8:0.1197, t9:0.1382
   📈 State Log -> t1:0.0516, t2:0.0781, t3:0.0945, t4:0.1075, t5:0.1196, t6:0.1325, t7:0.1466, t8:0.1631, t9:0.1816, t10:0.2026
   📍 Pos Log   -> t2:0.1430, t3:0.2306, t4:0.3047, t5:0.3528, t6:0.3667, t7:0.3524, t8:0.3225, t9:0.3016, t10:0.3207, t11:0.4165
🌟 [Best Updated] Val Loss: 0.757307 -> Saved to deep_dynamics_best_step_10.pth


Epoch 120 (Steps: 10): 100%|██████████| 211/211 [00:12<00:00, 17.18it/s]



[Epoch 120 | Unroll: 10] Train Loss: 0.416852, Val Loss: 0.756280
   ⚙️ Accel Log -> t0:1.4840, t1:0.3321, t2:0.1886, t3:0.1416, t4:0.1215, t5:0.1067, t6:0.1037, t7:0.1070, t8:0.1197, t9:0.1381
   📈 State Log -> t1:0.0516, t2:0.0781, t3:0.0945, t4:0.1075, t5:0.1195, t6:0.1323, t7:0.1464, t8:0.1628, t9:0.1813, t10:0.2022
   📍 Pos Log   -> t2:0.1430, t3:0.2308, t4:0.3051, t5:0.3533, t6:0.3675, t7:0.3533, t8:0.3233, t9:0.3022, t10:0.3208, t11:0.4160
🌟 [Best Updated] Val Loss: 0.756280 -> Saved to deep_dynamics_best_step_10.pth
